# Step 1: Data Preparation

In [ ]:
import sys, site
print("python:", sys.executable)
print("venv:", sys.prefix)
print("site-packages:", site.getsitepackages() if hasattr(site, "getsitepackages") else "n/a")

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

## Task 1: Download and load nhanes.csv

In [ ]:
# Load nhanes.csv
nhanes = pd.read_csv('../datasets/nhanes.csv')

# Task 1a
nhanes.head()

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDRETH1,DMQMILIZ,DMDBORN4,DMDYRUSR,DMDEDUC2,...,HIQ032F,HIQ032H,HIQ032I,HIQ210,INDFMMPC,INQ300,IND310,BPXOSY1,BPXODI1,LBXTC
0,132141.0,12.0,2.0,2.0,56.0,4.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,2.0,3.0,1.0,NaN,115.0,70.0,205.0
1,136977.0,12.0,2.0,1.0,29.0,3.0,2.0,2.0,2.0,5.0,...,NaN,NaN,NaN,2.0,3.0,2.0,2.0,125.0,71.0,193.0
2,134280.0,12.0,2.0,2.0,25.0,3.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,2.0,3.0,1.0,NaN,111.0,69.0,87.0
3,132522.0,12.0,2.0,2.0,64.0,3.0,2.0,1.0,NaN,4.0,...,NaN,NaN,NaN,2.0,NaN,NaN,NaN,113.0,90.0,215.0
4,138725.0,12.0,2.0,2.0,28.0,3.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,2.0,3.0,1.0,NaN,114.0,69.0,215.0


In [ ]:
# Task 1b
nhanes.info()

<class 'pandas.DataFrame'>
RangeIndex: 5703 entries, 0 to 5702
Data columns (total 33 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      5703 non-null   float64
 1   SDDSRVYR  5703 non-null   float64
 2   RIDSTATR  5703 non-null   float64
 3   RIAGENDR  5703 non-null   float64
 4   RIDAGEYR  5703 non-null   float64
 5   RIDRETH1  5703 non-null   float64
 6   DMQMILIZ  5703 non-null   float64
 7   DMDBORN4  5703 non-null   float64
 8   DMDYRUSR  1140 non-null   float64
 9   DMDEDUC2  5449 non-null   float64
 10  DMDMARTZ  5448 non-null   float64
 11  DMDHHSIZ  5703 non-null   float64
 12  WTINT2YR  5703 non-null   float64
 13  WTMEC2YR  5703 non-null   float64
 14  ALQ111    4929 non-null   float64
 15  ALQ121    4429 non-null   float64
 16  ALQ130    3657 non-null   float64
 17  ALQ151    4409 non-null   float64
 18  ALQ170    2113 non-null   float64
 19  HIQ011    5703 non-null   float64
 20  HIQ032A   3037 non-null   float64
 21  HI

## Task 3: Create BP and CHOL target features

a. **Blood pressure (BP)**: normal (systolic < 120 AND diastolic < 80) vs abnormal.  
b. **Cholesterol (CHOL)**: normal (<200), borderline (200–239), high (≥240).  
Drop BPXOSY1, BPXODI1, LBXTC after creating targets. 

In [ ]:
# Sentinel value used in NHANES for "Refused"/"Don't know" — treat as NaN
SENTINEL = 5.397605346934028e-79

# Replace sentinel with NaN before computing targets
nhanes_clean = nhanes.replace(SENTINEL, np.nan)

# Task 3a: Blood pressure — normal: systolic < 120 AND diastolic < 80
def compute_bp(row):
    sy, di = row['BPXOSY1'], row['BPXODI1']
    if pd.isna(sy) or pd.isna(di):
        return np.nan
    return 'normal' if (sy < 120 and di < 80) else 'abnormal'

nhanes_clean['BP'] = nhanes_clean.apply(compute_bp, axis=1)

# Task 3b: Cholesterol — normal (<200), borderline (200–239), high (≥240)
def compute_chol(val):
    if pd.isna(val):
        return np.nan
    if val < 200:
        return 'normal'
    if val <= 239:
        return 'borderline'
    return 'high'

nhanes_clean['CHOL'] = nhanes_clean['LBXTC'].apply(compute_chol)

# Drop source columns
nhanes_clean = nhanes_clean.drop(columns=['BPXOSY1', 'BPXODI1', 'LBXTC'])

nhanes_clean[['BP', 'CHOL']].head()

,BP,CHOL
0,normal,borderline
1,abnormal,normal
2,normal,normal
3,abnormal,borderline
4,normal,borderline


In [5]:
# Enforce column order per spec (32 columns)
REQUIRED_COLS = [
    'SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1',
    'DMQMILIZ', 'DMDBORN4', 'DMDYRUSR', 'DMDEDUC2', 'DMDMARTZ', 'DMDHHSIZ',
    'WTINT2YR', 'WTMEC2YR', 'ALQ111', 'ALQ121', 'ALQ130', 'ALQ151', 'ALQ170',
    'HIQ011', 'HIQ032A', 'HIQ032B', 'HIQ032D', 'HIQ032F', 'HIQ032H', 'HIQ032I',
    'HIQ210', 'INDFMMPC', 'INQ300', 'IND310', 'BP', 'CHOL'
]
nhanes_clean = nhanes_clean[REQUIRED_COLS]

## Task 4: Train/test split

In [ ]:
LASTNAME = 'cheng'

# BP dataset: drop rows where BP is missing
bp_cheng = nhanes_clean.dropna(subset=['BP']).copy()

bp_training_cheng, bp_test_cheng = train_test_split(
    bp_cheng, test_size=0.2, random_state=42
)

# CHOL dataset: drop rows where CHOL is missing
chol_cheng = nhanes_clean.dropna(subset=['CHOL']).copy()

chol_training_cheng, chol_test_cheng = train_test_split(
    chol_cheng, test_size=0.2, random_state=42
)

bp_training_cheng.head()
chol_training_cheng.head()

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDRETH1,DMQMILIZ,DMDBORN4,DMDYRUSR,DMDEDUC2,...,HIQ032D,HIQ032F,HIQ032H,HIQ032I,HIQ210,INDFMMPC,INQ300,IND310,BP,CHOL
5441,138701.0,12.0,2.0,1.0,75.0,3.0,1.0,1.0,NaN,5.0,...,NaN,6.0,NaN,NaN,2.0,3.0,1.0,NaN,abnormal,normal
4882,138727.0,12.0,2.0,2.0,73.0,5.0,2.0,2.0,6.0,4.0,...,NaN,NaN,NaN,NaN,2.0,3.0,1.0,NaN,abnormal,normal
3937,131032.0,12.0,2.0,2.0,18.0,3.0,2.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,2.0,3.0,1.0,NaN,abnormal,normal
1062,131250.0,12.0,2.0,2.0,18.0,4.0,2.0,1.0,NaN,NaN,...,4.0,NaN,NaN,NaN,2.0,1.0,2.0,1.0,normal,normal
4403,133160.0,12.0,2.0,1.0,75.0,3.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,NaN,2.0,3.0,1.0,NaN,abnormal,normal


## Task 5: Preprocessing pipelines plineb and plinec

In [ ]:
# Feature columns (exclude identifier and targets)
FEATURE_COLS = [c for c in REQUIRED_COLS if c not in ['SEQN', 'BP', 'CHOL']]

# Numeric features (continuous): age, weights, questionnaire counts
NUMERIC_COLS = ['RIDAGEYR', 'WTINT2YR', 'WTMEC2YR', 'DMDHHSIZ']
# Categorical/ordinal: rest of features (NHANES codes)
CATEGORICAL_COLS = [c for c in FEATURE_COLS if c not in NUMERIC_COLS]

def make_preprocessing_pipeline():
    """Pipeline: replace sentinel -> impute -> encode/scale."""
    numeric_transformer = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ])
    categorical_transformer = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    preprocess = ColumnTransformer([
        ('num', numeric_transformer, NUMERIC_COLS),
        ('cat', categorical_transformer, CATEGORICAL_COLS)
    ])
    return preprocess

plineb = make_preprocessing_pipeline()
plinec = make_preprocessing_pipeline()

# Fit and transform bp_training
X_bp = bp_training_cheng[FEATURE_COLS].replace(SENTINEL, np.nan)
plineb.fit(X_bp)
X_bp_transformed = plineb.transform(X_bp)
# Reconstruct as DataFrame for snapshot (optional; or use last 5 rows of array)
bp_transformed_df = pd.DataFrame(
    X_bp_transformed,
    index=bp_training_cheng.index
).assign(BP=bp_training_cheng['BP'].values)

bp_transformed_df.tail()

,0,1,2,3,4,5,6,7,8,9,...,112,113,114,115,116,117,118,119,120,BP
3906,0.685172,-0.764079,-0.824554,-0.420015,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,abnormal
5376,-0.353857,2.165311,1.990215,0.257171,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
5412,-0.080428,1.603543,1.854041,-1.097200,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
5579,-1.228829,-0.342594,-0.420396,-1.097200,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
891,-0.463228,-0.053945,0.839856,1.611542,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,abnormal


In [ ]:
# Fit and transform chol_training
X_chol = chol_training_cheng[FEATURE_COLS].replace(SENTINEL, np.nan)
plinec.fit(X_chol)
X_chol_transformed = plinec.transform(X_chol)
chol_transformed_df = pd.DataFrame(
    X_chol_transformed,
    index=chol_training_cheng.index
).assign(CHOL=chol_training_cheng['CHOL'].values)

chol_transformed_df.tail()

,0,1,2,3,4,5,6,7,8,9,...,110,111,112,113,114,115,116,117,118,CHOL
4900,-0.474714,-0.609762,-0.670187,-1.105386,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
519,0.078226,-0.094474,-0.345606,-0.426316,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
3435,-1.414711,0.646421,0.638253,0.252754,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,borderline
4193,0.852341,-0.868752,-0.932694,-1.105386,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,normal
967,-1.801768,0.545170,0.752616,-1.105386,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,normal


In [9]:
# Optional: save transformed data and pipelines for later use
# pd.DataFrame(X_bp_transformed).to_csv('bp_training_transformed.csv', index=False)
# pd.DataFrame(X_chol_transformed).to_csv('chol_training_transformed.csv', index=False)
# import joblib
# joblib.dump(plineb, 'plineb.joblib')
# joblib.dump(plinec, 'plinec.joblib')